# weld-qml-01-cnn-embeddings

**What this does.** For each of the 25 W0003 splits (same seeded grouping as the cut-paste notebook), it trains two ResNet-18s exactly as before:
- `cnn_real`: trained on the real training sections only
- `cnn_cp`: trained on real sections + 1,500 cut-paste defectives

It then saves the 512-number penultimate-layer embedding of every real section and of 600 cut-paste images, plus the CNN's own test scores. It also saves two split-independent feature sets: HOG (1,764-d) and a frozen ImageNet ResNet-18 (512-d, no fine-tuning).

**Settings:** Accelerator = GPU T4 (required: the notebook stops immediately without a GPU). Internet = ON (downloads the ImageNet weights).

**Inputs (Add Input):** `gdxray-w0003-labelled` and `gdxray-without-synthetic-for-defectd` (the same two datasets as the cut-paste notebook; the Stable Diffusion datasets are not needed).

**Outputs (/kaggle/working):** `qml_meta.csv`, `hog_real.npy`, `frozen_real.npy`, `emb/emb_r{rep}_f{fold}.npz` (25 files), `cnn_head_perfold.csv`.

**Runtime:** about 30–60 min on a T4 (50 ResNet-18 trainings of 600 steps each). Then click *Save Version → Save & Run All*, and use this notebook's output as the input of notebook 02.

In [ ]:
import builtins, functools, time
print = functools.partial(builtins.print, flush=True)
T_START = time.time()
def log(*a): print(f"[{time.time() - T_START:7.1f}s]", *a)
log("notebook started")

import os, socket, numpy as np, pandas as pd, cv2
from skimage.feature import hog
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, recall_score
log("libraries loaded")

def walk_find(name, want_dir=False, root="/kaggle/input", max_depth=6):
    hits, seen, base = [], set(), root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        real = os.path.realpath(dirpath)
        if real in seen or "Welds_Synthetic" in dirpath or dirpath.count("/") - base >= max_depth:
            dirnames[:] = []; continue
        seen.add(real)
        if name in (dirnames if want_dir else filenames): hits.append(os.path.join(dirpath, name))
    return sorted(hits)

def show_inputs(root="/kaggle/input", depth=4):
    base = root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        lvl = dirpath.count("/") - base
        if lvl >= depth: dirnames[:] = []; continue
        print("   " * lvl + os.path.basename(dirpath) + f"/  ({len(filenames)} files)")

BASE = "/kaggle/input/datasets/samarthgoel2604"
LABELS = f"{BASE}/gdxray-w0003-labelled/W0003_cell_labels.csv"
W1_DIR = f"{BASE}/gdxray-without-synthetic-for-defectd/Welds kaggle/Welds/W0001"
if not os.path.isfile(LABELS):
    log("labels not at the usual path, searching /kaggle/input ...")
    LABELS = (walk_find("W0003_cell_labels.csv") or [None])[0]
W3_DIR = os.path.join(os.path.dirname(LABELS), "W0003") if LABELS else None
if not os.path.isdir(W1_DIR):
    log("W0001 not at the usual path, searching /kaggle/input ...")
    W1_DIR = next((p for p in walk_find("W0001", want_dir=True)
                   if os.path.isfile(os.path.join(p, "W0001_0001.png")) and os.path.isdir(os.path.join(os.path.dirname(p), "W0002"))), None)
W2_DIR = os.path.join(os.path.dirname(W1_DIR), "W0002") if W1_DIR else None
if not (LABELS and W3_DIR and os.path.isdir(W3_DIR) and W1_DIR and os.path.isdir(W2_DIR)):
    print("\nCould not find the inputs. This is what is attached under /kaggle/input:"); show_inputs()
    raise FileNotFoundError("Add the datasets gdxray-w0003-labelled and gdxray-without-synthetic-for-defectd (Add Input)")
OUT = "/kaggle/working"
log("inputs found:"); print("  W0003 :", W3_DIR, f"({len(os.listdir(W3_DIR))} files)\n  labels:", LABELS, "\n  W0001 :", W1_DIR, "\n  W0002 :", W2_DIR)

SIZE, SEED = 224, 42
REPEATS, FOLDS, VAL_FRAC = 5, 5, 0.2
STEPS, BS, LR = 600, 32, 3e-4
N_CP = 1500
CP_KEEP = 600
PRETRAINED = True
ALLOW_CPU = False

In [ ]:
def weld_band(A, pad=0.3):
    p = np.median(A.astype(np.float32), axis=1)
    p = cv2.GaussianBlur(p.reshape(-1, 1), (1, 0), sigmaX=0, sigmaY=25).ravel()
    lo, hi = np.percentile(p, 10), np.percentile(p, 95)
    on = np.append(p > lo + 0.5 * (hi - lo), False)
    best, start = (0, 0, 0), 0
    for i in range(len(on)):
        if on[i] and (i == 0 or not on[i-1]): start = i
        if not on[i] and i > 0 and on[i-1] and i - start > best[0]: best = (i - start, start, i)
    _, a, b = best; h = b - a
    return max(0, int(a - pad * h)), min(A.shape[0], int(b + pad * h))

def rs(c): return cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)

W1_TO_W3 = {1: "W0003_0068.png", 2: "W0003_0011.png", 3: "W0003_0030.png", 4: "W0003_0036.png", 5: "W0003_0041.png",
            6: "W0003_0042.png", 7: "W0003_0012.png", 8: "W0003_0039.png", 9: "W0003_0006.png", 10: "W0003_0007.png"}
MIN_AREA, PADD = 15, 8

def extract_defects(w1_dir, w2_dir):
    out = []
    for k, film in W1_TO_W3.items():
        I = cv2.imread(os.path.join(w1_dir, f"W0001_{k:04d}.png"), 0)
        M = cv2.imread(os.path.join(w2_dir, f"W0002_{k:04d}.png"), 0)
        if I is None or M is None: continue
        if M.shape != I.shape: M = cv2.resize(M, (I.shape[1], I.shape[0]), interpolation=cv2.INTER_NEAREST)
        M = (M > 0).astype(np.uint8)
        n, lab, st, _ = cv2.connectedComponentsWithStats(M, 8)
        for c in range(1, n):
            x, y, w, h, a = st[c]
            if a < MIN_AREA or w > 400 or h > 200: continue
            x0, y0 = max(0, x - PADD), max(0, y - PADD)
            x1, y1 = min(I.shape[1], x + w + PADD), min(I.shape[0], y + h + PADD)
            patch = I[y0:y1, x0:x1]
            m = (lab[y0:y1, x0:x1] == c).astype(np.uint8)
            md = cv2.dilate(m, np.ones((5, 5), np.uint8))
            bg = cv2.inpaint(patch, md * 255, 5, cv2.INPAINT_TELEA).astype(np.float32)
            delta = (patch.astype(np.float32) - bg) * cv2.GaussianBlur(md.astype(np.float32), (5, 5), 0)
            out.append(dict(film=film, delta=delta, area=int(a)))
    return out

def paste(section, defects, rng, n_max=3):
    S = section.astype(np.float32); H, W = S.shape
    k = rng.integers(1, n_max + 1)
    for d in rng.choice(len(defects), k, replace=False):
        D = defects[d]["delta"]
        if rng.random() < 0.5: D = D[:, ::-1]
        if rng.random() < 0.5: D = D[::-1, :]
        D = D * rng.uniform(0.8, 1.2)
        h, w = D.shape
        if h >= H or w >= W:
            s = min((H - 2) / h, (W - 2) / w); D = cv2.resize(D, (max(1, int(w * s)), max(1, int(h * s)))); h, w = D.shape
        cy = int(rng.uniform(0.3, 0.7) * H); cx = int(rng.uniform(0, W))
        y0 = int(np.clip(cy - h // 2, 0, H - h)); x0 = int(np.clip(cx - w // 2, 0, W - w))
        S[y0:y0 + h, x0:x0 + w] += D
    return np.clip(S, 0, 255).astype(np.uint8)

def load_real():
    lab = pd.read_csv(LABELS)
    lab = lab[lab.clean_margin_ok == 1]
    X, N, y, g, meta = [], [], [], [], []
    for img, d in lab.groupby("image", sort=True):
        A = cv2.imread(os.path.join(W3_DIR, img), 0)
        if A is None: print("missing", img); continue
        a, b = weld_band(A)
        for _, r in d.iterrows():
            if r.x1 > A.shape[1]: continue
            sec = A[a:b, r.x0:r.x1].copy()
            N.append(sec); X.append(rs(sec)); y.append(int(r.label)); g.append(img)
            meta.append(dict(image=img, cell=int(r.cell), x0=int(r.x0), x1=int(r.x1), label=int(r.label)))
    return np.array(X, np.uint8), N, np.array(y), np.array(g), pd.DataFrame(meta)

def splits(groups):
    imgs = np.array(sorted(set(groups)))
    for rep in range(REPEATS):
        r = np.random.default_rng(SEED + rep)
        perm = r.permutation(imgs)
        fold_of = {im: i % FOLDS for i, im in enumerate(perm)}
        f = np.array([fold_of[x] for x in groups])
        for k in range(FOLDS):
            tr_imgs = sorted(set(groups[f != k]))
            val_imgs = set(r.choice(tr_imgs, max(1, int(round(VAL_FRAC * len(tr_imgs)))), replace=False))
            fit = np.where((f != k) & ~np.isin(groups, list(val_imgs)))[0]
            val = np.where(np.isin(groups, list(val_imgs)))[0]
            yield rep, k, fit, val, np.where(f == k)[0]

def best_threshold(yt, p):
    ts = np.unique(np.concatenate([p, [0.5]]))
    ba = [balanced_accuracy_score(yt, (p >= t).astype(int)) for t in ts]
    return float(ts[int(np.argmax(ba))])

def hog_feat(x):
    return hog(cv2.resize(x, (128, 128), interpolation=cv2.INTER_AREA) / 255.0, orientations=9,
               pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys").astype(np.float32)

In [ ]:
t0 = time.time()
Xr, Nr, yr, gr, meta = load_real()
meta.insert(0, "idx", np.arange(len(meta)))
meta.to_csv(f"{OUT}/qml_meta.csv", index=False)
log(f"REAL: {len(yr)} sections | defective {yr.sum()} | clean {(yr==0).sum()} | radiographs {len(set(gr))}")
DEF = extract_defects(W1_DIR, W2_DIR)
log(f"CUT-PASTE: {len(DEF)} real defects extracted from W0001")
H_real = np.stack([hog_feat(x) for x in Xr])
np.save(f"{OUT}/hog_real.npy", H_real)
log("HOG features:", H_real.shape)
SPLITS = list(splits(gr))
log(len(SPLITS), "splits | example sizes fit/val/test:", [(len(a), len(b), len(c)) for _, _, a, b, c in SPLITS[:3]])

In [ ]:
import torch, torch.nn as nn, torchvision
log(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
dev = "cpu"
if torch.cuda.is_available():
    log("GPU:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
    try:
        a = torch.randn(512, 512, device="cuda"); float((a @ a).sum()); dev = "cuda"
    except Exception as e:
        log("This GPU can't run the installed PyTorch:", repr(e)[:200])
if dev != "cuda" and not ALLOW_CPU:
    raise RuntimeError("No working GPU. Settings -> Accelerator -> GPU T4 x2 (not P100, not None), then run again. "
                       "On CPU this notebook needs more than Kaggle's 12-hour limit.")
log("device:", dev)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def prep(xb, train):
    x = torch.from_numpy(xb).to(dev).float().div(255).unsqueeze(1)
    if train:
        n = x.shape[0]
        x = torch.where(torch.rand(n, 1, 1, 1, device=dev) < 0.5, x.flip(3), x)
        x = torch.where(torch.rand(n, 1, 1, 1, device=dev) < 0.5, x.flip(2), x)
        mu = x.mean((2, 3), keepdim=True)
        x = (x - mu) * torch.empty(n, 1, 1, 1, device=dev).uniform_(0.85, 1.15) + mu
        x = x + torch.empty(n, 1, 1, 1, device=dev).uniform_(-0.06, 0.06)
        x = (x + torch.randn_like(x) * 0.01).clamp(0, 1)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

def make_model():
    w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None
    m = torchvision.models.resnet18(weights=w); m.fc = nn.Linear(512, 1)
    return m.to(dev)

def train(m, sources, steps, lr, rng, verbose=False):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    lossf = nn.BCEWithLogitsLoss(); m.train()
    pools = [(X, [np.where(y == 0)[0], np.where(y == 1)[0]], y) for X, y in sources]
    per = BS // (2 * len(pools))
    for _ in range(steps):
        xs, ts = [], []
        for X, (i0, i1), y in pools:
            idx = np.concatenate([rng.choice(i0, per), rng.choice(i1, per)])
            xs.append(X[idx]); ts.append(y[idx])
        xb = prep(np.concatenate(xs), True)
        tb = torch.from_numpy(np.concatenate(ts)).float().to(dev)
        with torch.autocast(device_type="cuda", enabled=dev == "cuda"):
            loss = lossf(m(xb).squeeze(1), tb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if verbose and (_ + 1) % 100 == 0: log(f"    first model: step {_ + 1}/{steps}")
    return m

@torch.no_grad()
def predict_and_embed(m, X):
    m.eval(); body = nn.Sequential(*list(m.children())[:-1]); P, E = [], []
    for i in range(0, len(X), 64):
        e = body(prep(X[i:i+64], False)).flatten(1).float()
        E.append(e.cpu().numpy()); P.append(torch.sigmoid(m.fc(e)).squeeze(1).cpu().numpy())
    return np.concatenate(P), np.concatenate(E).astype(np.float16)

socket.setdefaulttimeout(120)
log("downloading ImageNet ResNet-18 weights (needs Internet ON in Settings) ...")
try:
    frozen = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
except Exception as e:
    raise RuntimeError("Could not download the ImageNet weights. Turn Internet ON (Settings -> Internet) and run again.") from e
frozen.fc = nn.Linear(512, 1); frozen = frozen.to(dev)
_, F_real = predict_and_embed(frozen, Xr)
np.save(f"{OUT}/frozen_real.npy", F_real)
log("frozen ImageNet embeddings:", F_real.shape)

In [ ]:
os.makedirs(f"{OUT}/emb", exist_ok=True)
t_loop, n_start = time.time(), len(os.listdir(f"{OUT}/emb"))
rows = pd.read_csv(f"{OUT}/cnn_head_perfold.csv").to_dict("records") if os.path.exists(f"{OUT}/cnn_head_perfold.csv") else []
for rep, k, fit, val, te in SPLITS:
    path = f"{OUT}/emb/emb_r{rep}_f{k}.npz"
    if os.path.exists(path):
        log(f"rep {rep} fold {k}: already done, skipping"); continue
    rng = np.random.default_rng(SEED + 100 * rep + k)
    blocked = set(gr[val]) | set(gr[te])
    D = [d for d in DEF if d["film"] not in blocked]
    clean_fit = fit[yr[fit] == 0]
    src = rng.choice(clean_fit, N_CP)
    Xc = np.array([rs(paste(Nr[i], D, rng)) for i in src], np.uint8)
    CP = (np.concatenate([Xc, Xr[clean_fit]]), np.concatenate([np.ones(N_CP, int), np.zeros(len(clean_fit), int)]))
    keep = np.sort(np.random.default_rng(SEED + 7 + 100 * rep + k).choice(N_CP, CP_KEEP, replace=False))
    save = dict(fit=fit, val=val, te=te, cp_src=src[keep], n_defects_used=len(D),
                hog_cp=np.stack([hog_feat(x) for x in Xc[keep]]).astype(np.float16),
                frozen_cp=predict_and_embed(frozen, Xc[keep])[1])
    for name, sources in [("cnnreal", [(Xr[fit], yr[fit])]), ("cnncp", [(Xr[fit], yr[fit]), CP])]:
        torch.manual_seed(SEED + 100 * rep + k)
        t_m = time.time()
        m = train(make_model(), sources, STEPS, LR, rng, verbose=not rows)
        p_all, E_all = predict_and_embed(m, Xr)
        _, E_cp = predict_and_embed(m, Xc[keep])
        save[f"{name}_real"], save[f"{name}_cp"], save[f"prob_{name}"] = E_all, E_cp, p_all.astype(np.float32)
        thr = best_threshold(yr[val], p_all[val]); p, yt = p_all[te], yr[te]
        yp = (p >= thr).astype(int); rec = recall_score(yt, yp, average=None, labels=[0, 1], zero_division=0)
        row = dict(repeat=rep, fold=k, model=f"cnn_{name[3:]}", val_auc=roc_auc_score(yr[val], p_all[val]),
                   auc=roc_auc_score(yt, p), bal_acc=balanced_accuracy_score(yt, yp), rec_def=rec[1], rec_clean=rec[0], thr=thr)
        rows.append(row)
        log(f"rep {rep} fold {k} | {row['model']:8s} | AUC {row['auc']:.3f} | BA {row['bal_acc']:.3f} | model took {time.time()-t_m:.0f}s")
        if len(rows) == 1:
            took = time.time() - t_m
            log(f"    estimated total for 50 models: about {50 * took / 60:.0f} min" +
                ("  <-- far slower than a T4 (about 0.5 min per model): check that the GPU is really in use" if took > 180 else ""))
        del m; torch.cuda.empty_cache() if dev == "cuda" else None
    np.savez_compressed(path, **save)
    pd.DataFrame(rows).to_csv(f"{OUT}/cnn_head_perfold.csv", index=False)
    n_done = len(os.listdir(f"{OUT}/emb")); per = (time.time() - t_loop) / max(1, n_done - n_start)
    log(f"--- split {n_done}/{len(SPLITS)} saved | {per/60:.1f} min per split | about {per*(len(SPLITS)-n_done)/60:.0f} min left")

In [ ]:
df = pd.read_csv(f"{OUT}/cnn_head_perfold.csv")
print(df.groupby("model")[["auc", "bal_acc", "rec_def", "rec_clean"]].agg(["mean", "std"]).round(3))
print("\nPrevious notebook (for comparison): CNN real AUC 0.770, CNN +cut-paste AUC 0.848.")
print("Small differences are expected: GPU training is not bit-exact. The splits and cut-paste images are identical.")
print("Files:", sorted(os.listdir(OUT)), "| emb files:", len(os.listdir(f"{OUT}/emb")))